# [Step 1 - OpenAI and the provider-swap pattern] One chain, three engines

> **MLCourse - Agentic AI - Chat Models and Providers**

> Stage in the capstone: the generate stage - the final RAG chatbot is written once
> against LangChain's model interface, so you can develop it on free local Ollama and
> flip a single string to serve it with any cloud engine. This notebook proves that
> claim by running one identical chain on three providers.

### What you'll learn

- Minimal `ChatOpenAI` usage - this is the ONLY OpenAI notebook in the entire track.
- How `init_chat_model("provider:model")` turns provider choice into a config string.
- THE star lesson: build one LCEL chain once, execute it against ollama, groq, and
  openai without touching application code.
- A cost-conscious way to experiment with paid APIs (`max_tokens` budgeting).
- A final decision table for choosing providers per situation.

### Standard library imports


In [ ]:
import os                # Reads environment variables populated from .env below.
from pathlib import Path # Locates the track root folder.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv  # Standard track-wide key loader.

# Shared walk-up: resolve 03_agentic_ai/ and load its gitignored .env file.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Read both cloud keys up front; each demo checks the one it needs before calling.
OPENAI_KEY = os.getenv("OPENAI_API_KEY")
GROQ_KEY = os.getenv("GROQ_API_KEY")

# Jupyter plotting magic guarded so the file stays valid pure Python outside IPython.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("OpenAI key present:", bool(OPENAI_KEY))
print("Groq key present  :", bool(GROQ_KEY))


### 1. OpenAI in ninety seconds

OpenAI's GPT models are still the industry's quality reference point. Practical facts
for learners:

- Access requires a PAID account key from platform.openai.com stored as
  `OPENAI_API_KEY` in `03_agentic_ai/.env`.
- Billing is per token. `gpt-4o-mini` is priced for experimentation - but "cheap"
  is not "free", so we always set `max_tokens` in this course to bound spend.
- This track deliberately confines OpenAI to THIS notebook: everything else runs on
  Ollama or Groq so nobody needs a credit card to finish the course.

> **Common pitfall:** an unexpected 401 usually means an unpaid/expired account or a
> copied key with stray whitespace; a 429 means quota exhaustion - check usage on the
> platform dashboard before rewriting code.

In [2]:
if not OPENAI_KEY:
    print("[demo skipped] Add OPENAI_API_KEY to 03_agentic_ai/.env (see .env.example)")
else:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-4o-mini",     # Small, fast, inexpensive tier - ideal for coursework.
        api_key=OPENAI_KEY,
        temperature=0.3,
        max_tokens=100,          # Hard output cap = predictable worst-case cost per call.
    )
    try:
        print(llm.invoke("Say hi in five words").content)
    except Exception:
        print("[demo skipped] OpenAI call failed - check key validity, billing credit, network.")

Hello there! How are you?


### 2. The star section: one chain, three providers

Everything so far taught you three vendor classes with one shared interface. Now cash
that in. The plan:

1. Build a small RAG-flavored prompt template (module 02 teaches templates properly).
2. Compose it with ANY model using LCEL's pipe operator: `prompt | model` (module 04
   teaches chains properly). Today it is enough to read the pipe as "then feed to".
3. Swap models by changing ONE STRING passed to `init_chat_model` - nothing else moves.

If your app code only ever says `chain.invoke(question)`, then dev-local / prod-cloud /
benchmark-everywhere become deployment decisions, not rewrites.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

# The prompt is PROVIDER-AGNOSTIC: it is just text with a slot for the question.
PROMPT = ChatPromptTemplate.from_messages([
    # System role sets behavior once; every provider interprets it the same way.
    ("system", "You are a precise assistant. Answer in ONE short sentence."),
    ("human", "{question}"),        # Placeholder filled at invoke time.
])

def ask(provider_spec, question):
    """Run the SAME chain against whatever 'provider:model' string it is handed."""
    # Factory parses the prefix before the colon and constructs the matching class;
    # extra kwargs like temperature forward to whichever class gets built.
    model = init_chat_model(provider_spec, temperature=0.3)
    chain = PROMPT | model          # LCEL composition: format prompt, then call model.
    return chain.invoke({"question": question}).content   # Identical call shape everywhere.

Now sweep all three engines. Each entry pairs its provider string with the env var it
depends on, so missing keys produce informative skips instead of tracebacks - and at
least the Ollama row should run for everyone following along locally.

> **Pro tip:** make the provider a deployment setting, not a code edit:
> `PROVIDER = os.getenv("LLM_PROVIDER", "ollama:llama3.2")` lets ops swap engines
> without a code review. You will meet exactly this pattern in later modules.

In [4]:
PROVIDERS = [
    # (init_chat_model spec, required env var or None)
    ("ollama:llama3.2", None),                      # Local, no key - should work for everyone.
    ("groq:openai/gpt-oss-20b", "GROQ_API_KEY"),  # Free-tier key needed.
    ("openai:gpt-4o-mini", "OPENAI_API_KEY"),       # Paid key needed.
]

QUESTION = "In one sentence, why is retrieval-augmented generation useful?"

for provider_spec, required_key in PROVIDERS:
    print("=" * 60)
    print("Engine:", provider_spec)

    if required_key and not os.getenv(required_key):   # Key gate BEFORE any network attempt.
        print(f"[demo skipped] needs {required_key} in 03_agentic_ai/.env (see .env.example)")
        continue

    try:
        print("Answer:", ask(provider_spec, QUESTION))   # SAME function call for every engine.
    except Exception:
        # Ollama fails when the server/model is missing; clouds fail on network/quota.
        reason = ("Start Ollama, then run once: ollama pull llama3.2"
                  if provider_spec.startswith("ollama:")
                  else "check network access, key validity, and rate limits")
        print("[demo skipped]", reason)

print("=" * 60)
print("Notice: ask() never changed - only the provider string did.")

Engine: ollama:llama3.2


Answer: Retrieval-augmented generation combines the strengths of retrieval and generation models to leverage pre-trained knowledge and generate more accurate and informative outputs.
Engine: groq:openai/gpt-oss-20b


Answer: Retrieval‑augmented generation boosts accuracy and relevance by fetching up‑to‑date external facts before generating a response.
Engine: openai:gpt-4o-mini


Answer: Retrieval-augmented generation enhances the quality and relevance of generated responses by incorporating external information from a knowledge base.
Notice: ask() never changed - only the provider string did.


### 3. Why this abstraction earns its keep

| Situation | What the pattern gives you |
|---|---|
| Developing offline / on a plane | Point at `ollama:*`, zero cost, zero keys |
| Demos needing snappy replies | Flip to `groq:*` for extreme tokens/sec |
| Quality-sensitive production answer | Flip to `openai:*` if budget allows |
| Benchmarking engines honestly | Loop the same chain over several specs, compare outputs |
| Vendor incident / price change | Change one config value, not your codebase |

> **Common pitfall:** provider-specific parameters do NOT travel through the factory.
> ChatOllama caps output via `num_predict`, while Groq/OpenAI use `max_tokens` - pass
> such options conditionally per provider instead of assuming one name fits all.

### 4. Provider decision recap

| Criterion | Winner | Why |
|---|---|---|
| Zero cost, total privacy | Ollama | Runs locally; nothing leaves the machine |
| Raw speed on open models | Groq | LPU hardware serves hundreds of tokens/sec on a free tier |
| Model variety without downloads | HuggingFace hosted | Millions of repos callable over HTTPS |
| Reference quality baseline | OpenAI | GPT models remain the comparison point |

Rule of thumb for the rest of this track: **develop on Ollama, sprinkle Groq for speed,
reach for OpenAI only where quality justifies spend** - exactly how the capstone will be built.

### Summary & key takeaways

- `ChatOpenAI(model="gpt-4o-mini")` is the whole story of using OpenAI - and because
  it is metered, `max_tokens` discipline is part of good citizenship.
- `init_chat_model("provider:model")` converts vendor choice into a plain string, so
  configuration can live in environment variables rather than source code.
- One LCEL chain (`prompt | model`) ran unchanged on ollama, groq, and openai: THAT is
  the payoff of LangChain's uniform model interface.
- Guarded cells (key checks + try/except) kept the notebook green regardless of which
  providers you have - the same convention used across every module of this track.
- You now hold all four engines this track uses; module 02 takes control of WHAT goes
  into them with prompt templates, few-shotting, and partial variables.